# Agentic AI Resume Reviewer
## Multi-Step Workflow Using LangGraph + Google Gemini

### System Architecture

This notebook implements a **multi-agent agentic AI system** for resume analysis with iterative refinement.

**Agents:**
1. **Resume Parser Agent** – Extracts raw text from PDF/DOCX files
2. **Information Extractor Agent** – Uses Gemini LLM to extract structured data (name, contact, education, experience)
3. **Skills Analyzer Agent** – Matches resume skills against a target job description using LLM
4. **ATS Scorer Agent** – Computes an ATS (Applicant Tracking System) compatibility score
5. **Evaluator Agent** – Decides whether the resume meets quality threshold or needs refinement
6. **Improvement Agent** – Generates targeted, LLM-powered suggestions and refined bullet points

**Agentic Features:**
- State-based workflow via LangGraph `StateGraph`
- Conditional branching: if score < threshold → loop back to Improvement Agent
- Iterative refinement (up to 3 cycles)
- Each agent receives and enriches a shared state object

## Step 1: Imports & Setup

In [ ]:
import os
import re
import json
from typing import TypedDict, List
from pathlib import Path

import pandas as pd
from PyPDF2 import PdfReader
from docx import Document
from dotenv import load_dotenv
from google import genai

from langgraph.graph import StateGraph, END

load_dotenv()

# Configure Gemini (new google-genai SDK)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")
client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.0-flash"

def call_gemini(prompt: str) -> str:
    """Wrapper around Gemini API call."""
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return response.text.strip()

print("Setup complete. Gemini model:", GEMINI_MODEL)

## Step 2: Shared State Definition

All agents communicate through a single `ResumeState` TypedDict. Each agent reads from and writes back to this shared state — this is the backbone of the agentic workflow.

In [ ]:
class ResumeState(TypedDict):
    # Input
    file_path: str
    job_description: str

    # Agent 1: Parser
    raw_text: str

    # Agent 2: Extractor
    candidate_name: str
    email: str
    phone: str
    education: List[str]
    experience: List[str]

    # Agent 3: Skills Analyzer
    matched_skills: List[str]
    missing_skills: List[str]
    skill_match_pct: float

    # Agent 4: ATS Scorer
    ats_score: float
    score_breakdown: dict

    # Agent 5: Evaluator
    evaluation: str          # 'pass' or 'improve'
    quality_label: str       # 'Excellent' / 'Good' / 'Average' / 'Needs Improvement'
    iteration: int

    # Agent 6: Improvement
    suggestions: List[str]
    improved_bullets: List[str]
    final_feedback: str

print("ResumeState schema defined — 6 agent output sections")

## Step 3: Agent 1 — Resume Parser Agent

**Responsibility:** Read the resume file (PDF or DOCX) and extract plain text.  
**Input:** `file_path`  
**Output:** `raw_text`

In [ ]:
def resume_parser_agent(state: ResumeState) -> ResumeState:
    """Agent 1: Parses resume file into raw text."""
    print("[Agent 1] Resume Parser → Reading file...")
    file_path = state["file_path"]

    if not Path(file_path).exists():
        raise FileNotFoundError(f"Resume file not found: {file_path}")

    if file_path.lower().endswith(".pdf"):
        reader = PdfReader(file_path)
        text = "".join(page.extract_text() or "" for page in reader.pages)
    elif file_path.lower().endswith(".docx"):
        doc = Document(file_path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        raise ValueError("Unsupported file format. Use .pdf or .docx")

    print(f"   Extracted {len(text)} characters from {Path(file_path).name}")
    return {**state, "raw_text": text}

print("Agent 1 (Resume Parser) defined")

## Step 4: Agent 2 — Information Extractor Agent

**Responsibility:** Use Gemini LLM to extract structured information from raw resume text.  
**Input:** `raw_text`  
**Output:** `candidate_name`, `email`, `phone`, `education`, `experience`

In [ ]:
def information_extractor_agent(state: ResumeState) -> ResumeState:
    """Agent 2: Uses Gemini to extract structured info from resume text."""
    print("[Agent 2] Information Extractor → Calling Gemini...")

    prompt = f"""You are a resume parsing expert. Extract the following from the resume text.
Return ONLY a valid JSON object with these exact keys:
{{
  "candidate_name": "Full name or Unknown",
  "email": "email or Not Found",
  "phone": "phone number or Not Found",
  "education": ["degree at institution (year)"],
  "experience": ["role at company (duration)"]
}}

Resume Text:
{state['raw_text'][:4000]}

Return only the JSON, no markdown fences."""

    raw = call_gemini(prompt)
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        text = state["raw_text"]
        data = {
            "candidate_name": "Unknown",
            "email": (re.search(r'[\w\.-]+@[\w\.-]+', text) or type('', (), {'group': lambda *a: 'Not Found'})()).group(0),
            "phone": (re.search(r'(\+?\d[\d\-\s]{8,15})', text) or type('', (), {'group': lambda *a: 'Not Found'})()).group(0),
            "education": [],
            "experience": []
        }

    print(f"   Name: {data.get('candidate_name')} | Email: {data.get('email')} | Roles: {len(data.get('experience', []))}")
    return {
        **state,
        "candidate_name": data.get("candidate_name", "Unknown"),
        "email": data.get("email", "Not Found"),
        "phone": data.get("phone", "Not Found"),
        "education": data.get("education", []),
        "experience": data.get("experience", [])
    }

print("Agent 2 (Information Extractor) defined")

## Step 5: Agent 3 — Skills Analyzer Agent

**Responsibility:** Compare resume skills against a target job description using Gemini.  
**Input:** `raw_text`, `job_description`  
**Output:** `matched_skills`, `missing_skills`, `skill_match_pct`

In [ ]:
def skills_analyzer_agent(state: ResumeState) -> ResumeState:
    """Agent 3: Compares resume skills to job description using Gemini."""
    print("[Agent 3] Skills Analyzer → Matching skills with job description...")

    prompt = f"""You are a technical recruiter. Analyze the resume and job description.
Return ONLY a valid JSON object:
{{
  "matched_skills": ["skills present in BOTH resume and job description"],
  "missing_skills": ["skills required by JD but NOT in resume"]
}}

Job Description:
{state['job_description']}

Resume:
{state['raw_text'][:3000]}

Return only the JSON, no markdown."""

    raw = re.sub(r"```json|```", "", call_gemini(prompt)).strip()

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = {"matched_skills": [], "missing_skills": []}

    matched = data.get("matched_skills", [])
    missing = data.get("missing_skills", [])
    total = len(matched) + len(missing)
    pct = round((len(matched) / total * 100) if total > 0 else 0, 1)

    print(f"   Matched: {len(matched)} | Missing: {len(missing)} | Match rate: {pct}%")
    return {**state, "matched_skills": matched, "missing_skills": missing, "skill_match_pct": pct}

print("Agent 3 (Skills Analyzer) defined")

## Step 6: Agent 4 — ATS Scorer Agent

**Responsibility:** Compute a weighted ATS compatibility score from all extracted signals.  

| Component | Max Points |
|-----------|------------|
| Email present | 10 |
| Phone present | 10 |
| Education listed | 15 |
| Experience listed | 15 |
| Skill match % | 50 |

In [ ]:
def ats_scorer_agent(state: ResumeState) -> ResumeState:
    """Agent 4: Computes a weighted ATS score from extracted signals."""
    print("[Agent 4] ATS Scorer → Computing score...")

    breakdown = {}
    score = 0

    email_pts = 10 if state.get("email", "Not Found") != "Not Found" else 0
    phone_pts = 10 if state.get("phone", "Not Found") != "Not Found" else 0
    breakdown["Email Present"] = email_pts
    breakdown["Phone Present"] = phone_pts
    score += email_pts + phone_pts

    edu_pts = min(len(state.get("education", [])) * 8, 15)
    breakdown["Education"] = edu_pts
    score += edu_pts

    exp_pts = min(len(state.get("experience", [])) * 5, 15)
    breakdown["Experience"] = exp_pts
    score += exp_pts

    skill_pts = round(state.get("skill_match_pct", 0) * 0.5, 1)
    breakdown["Skill Match"] = skill_pts
    score += skill_pts

    final_score = round(min(score, 100), 1)
    print(f"   ATS Score: {final_score}/100 | Breakdown: {breakdown}")
    return {**state, "ats_score": final_score, "score_breakdown": breakdown}

print("Agent 4 (ATS Scorer) defined")

## Step 7: Agent 5 — Evaluator Agent (The Decision Maker)

**Responsibility:** Evaluate the ATS score and decide whether to pass or trigger an improvement loop.  
**This is the core agentic decision node** — it drives the feedback/iteration mechanism.

- Score ≥ 75 → `pass` (workflow ends)
- Score < 75 AND iteration < 3 → `improve` (loop back to Improvement Agent)
- Iteration ≥ 3 → `pass` (force exit to prevent infinite loop)

In [ ]:
PASS_THRESHOLD = 75
MAX_ITERATIONS = 3

def evaluator_agent(state: ResumeState) -> ResumeState:
    """Agent 5: Decides pass/improve based on ATS score and iteration count."""
    print("[Agent 5] Evaluator → Assessing quality...")

    score = state.get("ats_score", 0)
    iteration = state.get("iteration", 0)

    if score >= 80:   quality = "Excellent"
    elif score >= 65: quality = "Good"
    elif score >= 50: quality = "Average"
    else:             quality = "Needs Improvement"

    if score >= PASS_THRESHOLD or iteration >= MAX_ITERATIONS:
        decision = "pass"
        reason = "threshold met" if score >= PASS_THRESHOLD else "max iterations reached"
    else:
        decision = "improve"
        reason = f"score {score} < threshold {PASS_THRESHOLD}"

    print(f"   Quality: {quality} | Decision: {decision} ({reason}) | Cycle: {iteration}")
    return {**state, "evaluation": decision, "quality_label": quality, "iteration": iteration + 1}

def route_after_evaluation(state: ResumeState) -> str:
    """Conditional edge function: returns 'pass' or 'improve'."""
    return state["evaluation"]

print(f"Agent 5 (Evaluator) defined | Pass threshold: {PASS_THRESHOLD} | Max iterations: {MAX_ITERATIONS}")

## Step 8: Agent 6 — Improvement Agent

**Responsibility:** Use Gemini to generate specific, actionable suggestions and improved experience bullets.  
**After this agent runs**, flow returns to **Agent 4 (Scorer)** → **Agent 5 (Evaluator)** — creating the refinement loop.

In [ ]:
def improvement_agent(state: ResumeState) -> ResumeState:
    """Agent 6: Generates LLM-powered targeted suggestions and improved content."""
    cycle = state.get('iteration', 1)
    print(f"[Agent 6] Improvement Agent → Generating suggestions (refinement cycle {cycle})...")

    missing = state.get("missing_skills", [])
    score = state.get("ats_score", 0)
    quality = state.get("quality_label", "Average")
    experience = state.get("experience", [])

    prompt = f"""You are an expert resume coach. A resume scored {score}/100 (quality: {quality}).
Missing skills from job description: {', '.join(missing) if missing else 'None identified'}
Current experience entries: {'; '.join(experience) if experience else 'Not extracted'}

Return ONLY valid JSON:
{{
  "suggestions": [
    "Specific actionable suggestion 1",
    "Specific actionable suggestion 2",
    "Specific actionable suggestion 3",
    "Specific actionable suggestion 4",
    "Specific actionable suggestion 5"
  ],
  "improved_bullets": [
    "Achievement-oriented bullet with metrics 1",
    "Achievement-oriented bullet with metrics 2",
    "Achievement-oriented bullet with metrics 3"
  ],
  "overall_feedback": "2-3 sentence priority assessment"
}}

Tailor suggestions to the specific missing skills and score gap. No markdown."""

    raw = re.sub(r"```json|```", "", call_gemini(prompt)).strip()

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = {
            "suggestions": [
                f"Add missing skills to your skills section: {', '.join(missing[:3]) if missing else 'review JD'}",
                "Quantify every achievement with numbers and percentages",
                "Add a professional summary targeting this specific role",
                "List relevant certifications and personal projects",
                "Ensure name, email and phone are on the first line"
            ],
            "improved_bullets": [
                "Led a team of 8 engineers to deliver a microservices migration 20% ahead of schedule",
                "Reduced API response time by 40% via Redis caching and query optimization",
                "Increased CI/CD pipeline reliability from 78% to 97% using GitHub Actions"
            ],
            "overall_feedback": "Focus on quantifying achievements and closing the identified skill gaps to improve ATS scoring."
        }

    # Simulate projected improvement: after suggestions, skill match increases
    new_skill_match = min(state.get("skill_match_pct", 0) + 15, 100)

    print(f"   Generated {len(data.get('suggestions', []))} suggestions | Projected skill match: {new_skill_match}%")
    return {
        **state,
        "suggestions": data.get("suggestions", []),
        "improved_bullets": data.get("improved_bullets", []),
        "final_feedback": data.get("overall_feedback", ""),
        "skill_match_pct": new_skill_match
    }

print("Agent 6 (Improvement Agent) defined")

## Step 9: Build the LangGraph Workflow

Wire all 6 agents into a `StateGraph` with a conditional edge for the improvement loop.

```
START
  │
  ▼
Agent 1: Parser
  │
  ▼
Agent 2: Extractor
  │
  ▼
Agent 3: Skills Analyzer
  │
  ▼
Agent 4: ATS Scorer ◄──────────────┐
  │                                │
  ▼                                │
Agent 5: Evaluator                 │
  │                                │
  ├── score ≥ 75 ──► END           │
  │                                │
  └── score < 75 ──► Agent 6: Improvement ──┘
```

In [ ]:
workflow = StateGraph(ResumeState)

# Register all agent nodes
workflow.add_node("parser",      resume_parser_agent)
workflow.add_node("extractor",   information_extractor_agent)
workflow.add_node("skills",      skills_analyzer_agent)
workflow.add_node("scorer",      ats_scorer_agent)
workflow.add_node("evaluator",   evaluator_agent)
workflow.add_node("improvement", improvement_agent)

# Linear flow through the first 5 nodes
workflow.set_entry_point("parser")
workflow.add_edge("parser",    "extractor")
workflow.add_edge("extractor", "skills")
workflow.add_edge("skills",    "scorer")
workflow.add_edge("scorer",    "evaluator")

# Conditional branch at evaluator
workflow.add_conditional_edges(
    "evaluator",
    route_after_evaluation,
    {"pass": END, "improve": "improvement"}
)

# Improvement loops back to scorer for re-evaluation
workflow.add_edge("improvement", "scorer")

app = workflow.compile()
print("LangGraph workflow compiled")
print("Flow: parser → extractor → skills → scorer → evaluator")
print("Loop: evaluator --[improve]--> improvement → scorer (max 3 cycles)")

## Step 10: Create a Sample Resume for Demo

In [ ]:
def create_sample_resume():
    """Creates a sample resume DOCX for demonstration."""
    doc = Document()
    doc.add_heading("Alex Johnson", 0)
    doc.add_paragraph("Email: alex.johnson@email.com | Phone: +1-555-234-5678")
    doc.add_paragraph("LinkedIn: linkedin.com/in/alexjohnson | GitHub: github.com/alexj")

    doc.add_heading("Professional Summary", level=1)
    doc.add_paragraph(
        "Software Engineer with 4 years of experience building scalable backend systems. "
        "Proficient in Python, REST APIs, and AWS cloud infrastructure."
    )

    doc.add_heading("Education", level=1)
    doc.add_paragraph("B.S. Computer Science — State University (2020)")

    doc.add_heading("Experience", level=1)
    doc.add_paragraph("Software Engineer — TechCorp Inc. (2020–2024)")
    doc.add_paragraph("• Built REST APIs serving 500K daily requests using Python and FastAPI")
    doc.add_paragraph("• Managed PostgreSQL databases and optimized complex SQL queries")
    doc.add_paragraph("• Deployed applications on AWS EC2 and S3")
    doc.add_paragraph("• Containerized services using Docker")

    doc.add_heading("Skills", level=1)
    doc.add_paragraph("Python, FastAPI, PostgreSQL, SQL, AWS, Git, Linux, REST APIs, Docker")

    path = "sample_resume.docx"
    doc.save(path)
    print(f"Sample resume created: {path}")
    return path

RESUME_PATH = create_sample_resume()

## Step 11: Define Job Description & Run the Agentic Workflow

In [ ]:
JOB_DESCRIPTION = """
Senior Software Engineer — Cloud & Backend

Required Skills:
- Python, FastAPI or Django
- AWS (EC2, Lambda, S3, RDS)
- Docker and Kubernetes
- PostgreSQL and Redis
- CI/CD pipelines (GitHub Actions, Jenkins)
- Microservices architecture
- Terraform or CloudFormation
- REST API design and documentation
- Git, Linux
- Strong communication and documentation skills

Nice to have: GraphQL, Kafka, Prometheus/Grafana monitoring
"""

initial_state: ResumeState = {
    "file_path": RESUME_PATH,
    "job_description": JOB_DESCRIPTION,
    "raw_text": "",
    "candidate_name": "",
    "email": "",
    "phone": "",
    "education": [],
    "experience": [],
    "matched_skills": [],
    "missing_skills": [],
    "skill_match_pct": 0.0,
    "ats_score": 0.0,
    "score_breakdown": {},
    "evaluation": "",
    "quality_label": "",
    "iteration": 0,
    "suggestions": [],
    "improved_bullets": [],
    "final_feedback": ""
}

print("=" * 55)
print(" STARTING AGENTIC RESUME REVIEWER WORKFLOW")
print("=" * 55)

final_state = app.invoke(initial_state)

print("\n" + "=" * 55)
print(" WORKFLOW COMPLETE")
print("=" * 55)

## Step 12: Display Final Results Report

In [ ]:
print("\n" + "━" * 55)
print(" RESUME ANALYSIS REPORT")
print("━" * 55)

print(f"\nCandidate:  {final_state['candidate_name']}")
print(f"Email:      {final_state['email']}")
print(f"Phone:      {final_state['phone']}")

print(f"\nEducation:")
for e in final_state.get('education', []):
    print(f"  • {e}")

print(f"\nExperience:")
for ex in final_state.get('experience', []):
    print(f"  • {ex}")

print(f"\nMatched Skills ({len(final_state['matched_skills'])}):")
print(f"  {', '.join(final_state['matched_skills']) or 'None'}")

print(f"\nMissing Skills ({len(final_state['missing_skills'])}):")
print(f"  {', '.join(final_state['missing_skills']) or 'None'}")

print(f"\nSkill Match:  {final_state['skill_match_pct']}%")
print(f"ATS Score:    {final_state['ats_score']}/100  ({final_state['quality_label']})")
print("Score Breakdown:")
for k, v in final_state['score_breakdown'].items():
    print(f"  • {k}: {v} pts")

print(f"\nRefinement Cycles Run: {max(0, final_state['iteration'] - 1)}")

if final_state.get('suggestions'):
    print(f"\nImprovement Suggestions:")
    for i, s in enumerate(final_state['suggestions'], 1):
        print(f"  {i}. {s}")

if final_state.get('improved_bullets'):
    print(f"\nSample Improved Experience Bullets:")
    for b in final_state['improved_bullets']:
        print(f"  • {b}")

if final_state.get('final_feedback'):
    print(f"\nOverall Feedback:")
    print(f"  {final_state['final_feedback']}")

print("\n" + "━" * 55)

## Step 13: Summary Table

In [ ]:
df = pd.DataFrame({
    "Metric": [
        "Candidate Name", "Email", "Phone",
        "Education Entries", "Experience Entries",
        "Matched Skills", "Missing Skills",
        "Skill Match %", "ATS Score",
        "Quality Label", "Refinement Cycles"
    ],
    "Value": [
        final_state["candidate_name"],
        final_state["email"],
        final_state["phone"],
        len(final_state["education"]),
        len(final_state["experience"]),
        ", ".join(final_state["matched_skills"]) or "None",
        ", ".join(final_state["missing_skills"]) or "None",
        f"{final_state['skill_match_pct']}%",
        f"{final_state['ats_score']}/100",
        final_state["quality_label"],
        max(0, final_state["iteration"] - 1)
    ]
})

df

## Step 14: Project Report — Workflow Design & Decisions

### Problem Chosen
Resume reviewing is time-consuming and inconsistent when done manually. An agentic system can decompose this task into parallel concerns — parsing, understanding, matching, scoring, and refining — mirroring how an expert recruiter thinks step by step.

### System Design

| Agent | Role | Technology |
|-------|------|------------|
| 1. Parser | File → Plain Text | PyPDF2 / python-docx |
| 2. Extractor | Text → Structured JSON | Google Gemini LLM |
| 3. Skills Analyzer | Resume + JD → Match/Gap | Google Gemini LLM |
| 4. ATS Scorer | Signals → Numeric Score | Weighted rule formula |
| 5. Evaluator | Score → pass/improve decision | Threshold logic |
| 6. Improvement | Gaps → Suggestions + Bullets | Google Gemini LLM |

**Orchestration:** LangGraph `StateGraph` with a conditional edge at the Evaluator node.

### How the Agentic Workflow Operates

1. **Task Decomposition** — Resume review is split into 6 independent concerns. No single agent tries to do everything.

2. **Sequential State Passing** — A `ResumeState` TypedDict flows through every agent. Each agent reads the full current state and returns an enriched copy. No agent loses context from prior steps.

3. **State-Based Decision Making** — The Evaluator (Agent 5) is a routing node, not just a processing node. It reads `ats_score` and `iteration` and returns either `"pass"` or `"improve"`. The LangGraph conditional edge dispatches to `END` or `improvement` accordingly.

4. **Iterative Refinement Loop** — When score < 75, the Improvement Agent generates targeted suggestions, increases `skill_match_pct` (simulating the candidate applying the advice), and returns to the ATS Scorer for re-evaluation. This loop runs up to 3 times.

5. **Feedback Mechanism** — The ATS score is recomputed after every improvement cycle. The score progression across iterations is a concrete, measurable feedback signal driving the workflow forward.

### Key Results / Observations
- The LangGraph conditional edge is the critical agentic component — it converts a linear pipeline into a decision-making loop
- Gemini LLM handles all ambiguous NLP tasks (extraction, matching, generation); deterministic rule-based logic handles scoring — this hybrid approach is both reliable and intelligent
- The `MAX_ITERATIONS` cap prevents infinite loops while allowing meaningful refinement
- State accumulation ensures Agent 6 has full context from all prior agents when generating suggestions